New Notebook Created by Jupyter MCP Server

In [1]:
# XAI Brain Figures — Reproducible Notebook
# ============================================
# Builds:
#   1. fig_xai_brain_overlay_all23.{png,svg}      — continuous heatmap, all 23 channels (Zhang Fig 4C style)
#   2. fig_xai_brain_overlay_top5.{png,svg}       — heatmap from top-5 channels only (Zhang Fig 5 style)
#   3. fig_xai_brain_colormap_grid.{png,svg}      — colormap variety contact sheet
#   4. fig_montage_brain_3views.{png,svg}         — front + left + right channel-dot montage
#   5. fig_top5_hbo_bars.{png,svg}                — HC vs GAD HbO_avg bar charts (FDR-marked)
#
# Headline-XAI cell: ST × HbO × LOSO × mt2 native (per project memory)
# Reproducible: re-run all cells top-to-bottom in fresh kernel.

import os
os.environ["PYVISTA_OFF_SCREEN"] = "true"
os.environ["MPLBACKEND"] = "Agg"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import rcParams
import imageio.v3 as iio
from scipy.spatial.distance import cdist
import mne

mne.set_log_level("ERROR")
rcParams["font.family"] = ["DejaVu Sans", "Helvetica", "Arial", "sans-serif"]
rcParams["savefig.dpi"] = 300

REPO = Path('/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method')
ELC = REPO / 'data' / 'brainproducts-RNP-BA-128-custom.elc'
GNG_DIR = REPO / 'data' / 'raw' / 'anxiety' / 'EA062' / 'GNG'
FIG_DIR = REPO / 'research' / 'paper-materials' / 'figures'
INTERM = FIG_DIR / '_intermediate'
INTERM.mkdir(parents=True, exist_ok=True)
print("REPO:", REPO)
print("MNE:", mne.__version__)
mne.viz.set_3d_backend("notebook")
print("3D backend:", mne.viz.get_3d_backend())

REPO: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method
MNE: 1.12.0


3D backend: notebook


In [2]:
# ─── Setup: info + fsaverage + channel positions ───────────────────────
fs_root = mne.datasets.fetch_fsaverage(verbose=False)
subjects_dir = Path(fs_root).parent
surf_dir = subjects_dir / "fsaverage" / "surf"
verts_lh, _ = mne.read_surface(str(surf_dir / "lh.pial"))
verts_rh, _ = mne.read_surface(str(surf_dir / "rh.pial"))
print("fsaverage at:", fs_root)
print("pial verts: lh", verts_lh.shape, "rh", verts_rh.shape)

# Load info via notebook 2_'s recipe (Beer-Lambert → pick(hbo)).
raw = mne.io.read_raw_nirx(str(GNG_DIR), preload=False, verbose=False)
montage = mne.channels.read_custom_montage(str(ELC), head_size=0.0825)
raw.set_montage(montage)
raw_od = mne.preprocessing.nirs.optical_density(raw, verbose=False)
raw_haemo = mne.preprocessing.nirs.beer_lambert_law(raw_od, ppf=6.0)
info = raw_haemo.copy().pick_types(fnirs='hbo', verbose=False).info
print("n_channels:", len(info.ch_names))

# Channel midpoints in MRI mm (apply fsaverage head→MRI trans).
trans = mne.read_trans(str(subjects_dir / "fsaverage" / "bem" / "fsaverage-trans.fif"))
head_positions = np.array([ch["loc"][:3] for ch in info["chs"]])
mri_positions_mm = mne.transforms.apply_trans(trans["trans"], head_positions) * 1000.0
print("mri_positions_mm shape:", mri_positions_mm.shape)
print("sample S1_D1 (MRI mm):", mri_positions_mm[0])

fsaverage at: /home/user/mne_data/MNE-fsaverage-data/fsaverage
pial verts: lh (163842, 3) rh (163842, 3)
n_channels: 23
mri_positions_mm shape: (23, 3)
sample S1_D1 (MRI mm): [-18.53291117  69.45304574   6.67208229]


/tmp/ipykernel_1850571/266978638.py:11: RuntimeWarning: MNE has not been tested with Aurora version 2021.9.0-20-g15526401-dirty
  raw = mne.io.read_raw_nirx(str(GNG_DIR), preload=False, verbose=False)


In [3]:
# ─── Load XAI importance + ROI groups + top-5 ─────────────────────────
CHANNEL_NAMES = ['S1_D1','S1_D3','S2_D2','S2_D1','S2_D5','S3_D1','S3_D3','S3_D4','S3_D6',
                 'S4_D4','S4_D5','S4_D7','S5_D2','S5_D5','S5_D8','S6_D3','S6_D6',
                 'S7_D4','S7_D6','S7_D7','S8_D5','S8_D7','S8_D8']
N_CH = len(CHANNEL_NAMES)

xai_csv = REPO / 'research/xai/st/hbo/loso/mt2/native/node_importance.csv'
df_xai = pd.read_csv(xai_csv).set_index('channel')
importance = df_xai.loc[CHANNEL_NAMES, 'mean'].to_numpy()
ranks = df_xai.loc[CHANNEL_NAMES, 'rank'].to_numpy()
top5_idx = np.argsort(ranks)[:5]
top5_channels = [CHANNEL_NAMES[i] for i in top5_idx]
top5_pos = mri_positions_mm[top5_idx]
top5_imp = importance[top5_idx]
print("Top-5 channels:", top5_channels)

# ROI groups (notebook 2_'s indices for VMPFC / DMPFC / DLPFC).
vmpfc_idx = sorted({0, 3, 2})
dmpfc_idx = sorted({7, 17, 18, 9, 19, 10, 11, 21})
dlpfc_idx = sorted({6, 8, 16, 15, 13, 14, 22, 20})
default_idx = [i for i in range(N_CH)
               if i not in set(vmpfc_idx + dmpfc_idx + dlpfc_idx)]
fnirs_colors = np.tile(np.array([1.0, 0.647, 0.0]), (N_CH, 1))  # default = orange
for i in vmpfc_idx: fnirs_colors[i] = [0.0, 0.0, 1.0]   # blue
for i in dmpfc_idx: fnirs_colors[i] = [0.0, 1.0, 0.0]   # green
for i in dlpfc_idx: fnirs_colors[i] = [1.0, 0.0, 0.0]   # red
print("VMPFC=blue, DMPFC=green, DLPFC=red, default=orange")

Top-5 channels: ['S5_D5', 'S7_D4', 'S4_D4', 'S8_D5', 'S1_D1']
VMPFC=blue, DMPFC=green, DLPFC=red, default=orange


In [4]:
# ─── Helper: Gaussian-weighted vertex overlay + multi-view rendering ────
SIGMA_MM = 15.0

def compute_vertex_overlay(verts, ch_positions, ch_importance, sigma=SIGMA_MM):
    """value(v) = Σ_c imp_c · exp(-‖v - pos_c‖² / 2σ²)"""
    d2 = cdist(verts, ch_positions) ** 2
    weights = np.exp(-d2 / (2 * sigma * sigma))
    return weights @ ch_importance

def render_overlay_4views(values_lh, values_rh, stem, cmap='inferno',
                           threshold_pct=85):
    """Render 4-view brain overlay → 4 PNGs in INTERM dir; return paths dict."""
    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex="low_contrast", alpha=0.95, units='mm', size=(1100, 900),
    )
    pos_vals = np.concatenate([values_lh, values_rh])
    pos_vals = pos_vals[pos_vals > 0]
    fmin = np.percentile(pos_vals, threshold_pct)
    fmax = np.percentile(pos_vals, 99.5)
    fmid = (fmin + fmax) / 2
    b.add_data(values_lh, hemi='lh', colormap=cmap, smoothing_steps=10,
               fmin=fmin, fmid=fmid, fmax=fmax, colorbar=True, transparent=True)
    b.add_data(values_rh, hemi='rh', colormap=cmap, smoothing_steps=10,
               fmin=fmin, fmid=fmid, fmax=fmax, colorbar=False, transparent=True)
    view_specs = {
        'rostral': dict(view='rostral'),
        'dorsal':  dict(view='dorsal'),
        'lat_lh':  dict(azimuth=180, elevation=90, hemi='lh'),
        'lat_rh':  dict(azimuth=0,   elevation=90, hemi='rh'),
    }
    paths = {}
    for name, kw in view_specs.items():
        try: b.show_view(**kw)
        except Exception: b.show_view()
        p = INTERM / f"{stem}_{name}.png"
        b.save_image(str(p), mode='rgba')
        paths[name] = p
    b.close()
    return paths

# Vertex-overlay arrays — compute once, reuse for colormap variety.
print("Computing all-23 overlay...")
values_lh_all = compute_vertex_overlay(verts_lh, mri_positions_mm, importance)
values_rh_all = compute_vertex_overlay(verts_rh, mri_positions_mm, importance)
print("Computing top-5 overlay...")
values_lh_top5 = compute_vertex_overlay(verts_lh, top5_pos, top5_imp)
values_rh_top5 = compute_vertex_overlay(verts_rh, top5_pos, top5_imp)
print("Done.")

Computing all-23 overlay...
Computing top-5 overlay...
Done.


In [5]:
# ─── Section 1: Colormap variety contact sheet ────────────────────────
# Render rostral view for each candidate colormap so you can compare.
CMAPS = ['inferno', 'magma', 'plasma', 'viridis',
         'hot',     'YlOrRd', 'OrRd',  'Reds',
         'RdYlBu_r','Spectral_r','jet','turbo']

for cmap in CMAPS:
    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex="low_contrast", alpha=0.95, units='mm', size=(900, 750),
    )
    pos = np.concatenate([values_lh_all, values_rh_all])
    pos = pos[pos > 0]
    fmin = np.percentile(pos, 85); fmax = np.percentile(pos, 99.5)
    fmid = (fmin + fmax) / 2
    b.add_data(values_lh_all, hemi='lh', colormap=cmap, smoothing_steps=10,
               fmin=fmin, fmid=fmid, fmax=fmax, colorbar=False, transparent=True)
    b.add_data(values_rh_all, hemi='rh', colormap=cmap, smoothing_steps=10,
               fmin=fmin, fmid=fmid, fmax=fmax, colorbar=False, transparent=True)
    b.show_view(view='rostral')
    b.save_image(str(INTERM / f"cmap_grid_{cmap}.png"), mode='rgba')
    b.close()
    print(f"  rendered: {cmap}")

# Composite into a 3x4 grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for ax, cmap in zip(axes.flat, CMAPS):
    img = mpimg.imread(str(INTERM / f"cmap_grid_{cmap}.png"))
    ax.imshow(img)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f"cmap='{cmap}'", fontsize=12, fontweight='bold')
fig.suptitle("Colormap variety — ST × HbO × LOSO × mt2 all-23 (rostral)",
              fontsize=14, fontweight='bold', y=0.995)
fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.01,
                     wspace=0.04, hspace=0.10)
out = FIG_DIR / 'fig_xai_brain_colormap_grid.png'
fig.savefig(str(out), dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(str(out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
plt.close(fig)
print("Saved:", out.name)

2026-05-12 05:42:55.606 (  68.424s) [    78FFF15AE740]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


  rendered: inferno


  rendered: magma


  rendered: plasma


  rendered: viridis


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


Saved: fig_xai_brain_colormap_grid.png


In [14]:
# ─── Section 2: 3-view montage (Front + Left + Right) ─────────────────
# v6 iteration:
#   - per-view label filtering (lateral views: hemi-only + midline channels)
#   - pure UP (+Z) label offset → no overlap with dots in any view
#   - save EACH view as its own individual figure + the 1x3 composite

cluster_centre = mri_positions_mm.mean(axis=0)
shrink_factor = 0.85
base_shrunk = cluster_centre + (mri_positions_mm - cluster_centre) * shrink_factor
mri_pos_dots = base_shrunk.copy()
mri_pos_dots[:, 2] -= 20.0

inv_trans = np.linalg.inv(trans["trans"])
head_positions_dots_m = mne.transforms.apply_trans(inv_trans, mri_pos_dots / 1000.0)
info_dots = info.copy()
for i, ch in enumerate(info_dots["chs"]):
    new_loc = ch["loc"].copy()
    new_loc[:3] = head_positions_dots_m[i]
    ch["loc"] = new_loc

labels = [str(i + 1) for i in range(N_CH)]

# Pure +Z (dorsal) label offset — works for all three views because the
# screen-up axis is world-Z in rostral / left-lateral / right-lateral.
LABEL_UP_MM = 5.0
label_offset = np.zeros_like(mri_pos_dots)
label_offset[:, 2] = LABEL_UP_MM
label_positions = mri_pos_dots + label_offset

# Hemisphere selection by MRI x-coord (+x = subject's right, -x = left).
MIDLINE_MM = 8.0
x = mri_positions_mm[:, 0]
mask_lh_view = x <  MIDLINE_MM
mask_rh_view = x > -MIDLINE_MM
mask_all     = np.ones(N_CH, dtype=bool)

print(f"  Frontal:   {mask_all.sum():2d} labels (all)")
print(f"  Lat-LH:    {mask_lh_view.sum():2d} labels (LH + midline)")
print(f"  Lat-RH:    {mask_rh_view.sum():2d} labels (RH + midline)")

def render_one_view(stem, view_name, view_kwargs, label_mask):
    b = mne.viz.Brain(
        "fsaverage", subjects_dir=str(subjects_dir), background="white",
        cortex="low_contrast", alpha=1.0, units='mm', size=(1100, 950),
    )
    b.add_sensors(info_dots, trans="fsaverage", fnirs=['channels'],
                   sensor_scales=6.0,
                   sensor_colors={'fnirs': fnirs_colors})
    pos_subset = label_positions[label_mask]
    lab_subset = [l for l, m in zip(labels, label_mask) if m]
    b.plotter.add_point_labels(
        pos_subset, lab_subset,
        font_size=28, text_color="black",
        point_color="white", point_size=1,
        show_points=False, always_visible=True,
        shape=None, margin=0, name=f"ch_numbers_{view_name}",
    )
    try: b.show_view(**view_kwargs)
    except Exception: b.show_view()
    p = INTERM / f"{stem}_{view_name}.png"
    b.save_image(str(p), mode='rgba')
    b.close()
    return p

stem = 'montage_3view'
paths_montage = {}
paths_montage['frontal'] = render_one_view(stem, 'frontal',
    dict(view='rostral'), mask_all)
paths_montage['lat_lh']  = render_one_view(stem, 'lat_lh',
    dict(azimuth=180, elevation=90, hemi='lh'), mask_lh_view)
paths_montage['lat_rh']  = render_one_view(stem, 'lat_rh',
    dict(azimuth=0,   elevation=90, hemi='rh'), mask_rh_view)

def crop_white(img_path, margin=8):
    img = iio.imread(str(img_path))
    rgb = img[:, :, :3].astype(np.int32)
    sat = rgb.max(axis=2) - rgb.min(axis=2)
    is_content = (sat > 30) | (rgb.sum(axis=2) < 600)
    ys, xs = np.where(is_content)
    if len(ys) == 0: return img
    y0, y1 = max(0, ys.min()-margin), min(img.shape[0], ys.max()+margin)
    x0, x1 = max(0, xs.min()-margin), min(img.shape[1], xs.max()+margin)
    return img[y0:y1, x0:x1]

cropped_imgs = {k: crop_white(v) for k, v in paths_montage.items()}
for k, v in cropped_imgs.items():
    print(f"  {k:8s}: {v.shape}")

# Save EACH view as its own standalone figure (front / left / right) for
# flexible downstream layout.
single_titles = {
    'frontal': ('fig_montage_brain_front',       'Frontal'),
    'lat_lh':  ('fig_montage_brain_left',        'Left lateral'),
    'lat_rh':  ('fig_montage_brain_right',       'Right lateral'),
}
for key, (stem_out, title) in single_titles.items():
    fig_s, ax_s = plt.subplots(figsize=(6, 5.5))
    ax_s.imshow(cropped_imgs[key])
    ax_s.set_xticks([]); ax_s.set_yticks([])
    for s in ax_s.spines.values():
        s.set_visible(False)
    ax_s.set_title(title, fontsize=14, fontweight='bold')
    p_out = FIG_DIR / f"{stem_out}.png"
    fig_s.savefig(str(p_out), dpi=300, bbox_inches='tight', facecolor='white')
    fig_s.savefig(str(p_out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
    plt.close(fig_s)
    print("Saved:", p_out.name)

# Composite 1x3 (back-compat)
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
view_order = [('frontal', 'A — Frontal'),
              ('lat_lh',  'B — Left lateral'),
              ('lat_rh',  'C — Right lateral')]
for ax, (name, sub) in zip(axes, view_order):
    ax.imshow(cropped_imgs[name])
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.text(-0.02, 1.03, sub, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
plt.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.01, wspace=0.03)
out = FIG_DIR / 'fig_montage_brain_3views.png'
fig.savefig(str(out), dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(str(out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
plt.close(fig)
print("Saved:", out.name)

  Frontal:   23 labels (all)
  Lat-LH:    13 labels (LH + midline)
  Lat-RH:    13 labels (RH + midline)


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


Saved: fig_montage_brain_3views.png


In [7]:
# ─── Section 3: Top-5 HbO bar charts (HC vs GAD, FDR-marked) ──────────
# Per-channel HC/GAD HbO_avg statistics already computed in
# src/notebook/statistical-analysis/02_brain_activation/results_brain_activation_stats.csv
# (mean_hc, std_hc, mean_gad, std_gad, U, p_raw, d, p_fdr, sig_fdr).
# Group sizes: HC=33, GAD=29 (per project memory).

stats_csv = REPO / 'src/notebook/statistical-analysis/02_brain_activation/results_brain_activation_stats.csv'
df_stats = pd.read_csv(stats_csv)
df_stats = df_stats[df_stats['task'] == 'GNG'].set_index('channel')

N_HC, N_GAD = 33, 29
sem_hc_factor = 1.0 / np.sqrt(N_HC)
sem_gad_factor = 1.0 / np.sqrt(N_GAD)

# Per-channel Brodmann region (from atlas) for sub-titles
atlas_csv = REPO / 'research/xai/atlas/channel_to_brodmann.csv'
df_atlas = pd.read_csv(atlas_csv)
# Pick the most-probable BA per channel
def primary_ba(ch):
    sub = df_atlas[df_atlas['channel'] == ch].sort_values('probability', ascending=False)
    if sub.empty: return 'Unlabelled'
    r = sub.iloc[0]
    ba = r['ba_label'].replace('Brodmann.', 'BA')
    return f"{ba}-{r['hemi']}"

# Build bar-chart panel: 1 row × 5 columns for top-5 channels.
fig, axes = plt.subplots(1, 5, figsize=(15, 4.5), sharey=True)
bar_w = 0.6
for ax, ch in zip(axes, top5_channels):
    row = df_stats.loc[ch]
    means = [row['mean_hc'], row['mean_gad']]
    sems  = [row['std_hc'] * sem_hc_factor, row['std_gad'] * sem_gad_factor]
    bars = ax.bar([0, 1], means, yerr=sems,
                   width=bar_w,
                   color=['#5577CC', '#E74C3C'],
                   edgecolor='black', linewidth=0.8,
                   capsize=6, error_kw=dict(elinewidth=1.2, ecolor='black'))
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['HC', 'GAD'], fontsize=11)
    ax.set_title(f"{ch}\n({primary_ba(ch)})", fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='y', labelsize=10)

    # FDR significance marker
    p_fdr = row['p_fdr']
    sig_fdr = bool(row['sig_fdr'])
    d = row['d']
    sig_str = "*" if sig_fdr else "ns"
    # Place marker above the higher bar
    y_max = max(m + s for m, s in zip(means, sems))
    ax.annotate(sig_str, xy=(0.5, y_max * 1.04),
                ha='center', va='bottom', fontsize=14,
                fontweight='bold' if sig_fdr else 'normal',
                color='#000000' if sig_fdr else '#888888')
    ax.text(0.5, y_max * 1.12,
             f"p_fdr={p_fdr:.3g}\nd={d:+.2f}",
             ha='center', va='bottom', fontsize=8, color='#444444')

axes[0].set_ylabel("HbO concentration (a.u.)", fontsize=11, fontweight='bold')
fig.suptitle("Top-5 XAI channels — HC vs GAD HbO concentration (mean ± SEM, * = FDR-significant)",
              fontsize=13, fontweight='bold', y=1.04)
plt.subplots_adjust(wspace=0.2)

out = FIG_DIR / 'fig_top5_hbo_bars.png'
fig.savefig(str(out), dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(str(out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
plt.close(fig)
print("Saved:", out.name)

# Also print the values
print("\nTop-5 channel stats (HC vs GAD):")
print(f"  {'ch':7s}  {'BA':10s}  {'mean_HC':>9s}  {'mean_GAD':>9s}  {'d':>7s}  {'p_fdr':>9s}  sig_fdr")
for ch in top5_channels:
    r = df_stats.loc[ch]
    print(f"  {ch:7s}  {primary_ba(ch):10s}  {r['mean_hc']:9.4f}  {r['mean_gad']:9.4f}  {r['d']:+7.2f}  {r['p_fdr']:9.4f}  {bool(r['sig_fdr'])}")

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


Saved: fig_top5_hbo_bars.png

Top-5 channel stats (HC vs GAD):
  ch       BA            mean_HC   mean_GAD        d      p_fdr  sig_fdr
  S5_D5    BA46-R         0.9175     0.8849    -0.92     0.0139  True
  S7_D4    BA8-L          0.8720     0.8737    +0.03     0.8767  False
  S4_D4    BA9-R          0.9288     0.9217    -0.22     0.3837  False
  S8_D5    BA9-R          0.9027     0.8895    -0.33     0.3837  False
  S1_D1    BA10-L         0.9083     0.8684    -0.81     0.0139  True


In [15]:
# ─── Section 3 (v2): Top-5 HbO bar charts with per-subject strip plot ──
# Re-compute per-subject HbO STD activation (same recipe as
# 02_brain_activation_anova.ipynb's load_subject_activation) so we can
# overlay each subject's dot on the bar.

import os
DATA_DIR = REPO / 'data' / 'processed-new-mc'
TASK = 'GNG'
HC_IDS = sorted(os.listdir(DATA_DIR / TASK / 'healthy'))
GAD_IDS = sorted(os.listdir(DATA_DIR / TASK / 'anxiety'))
N_HC = len(HC_IDS); N_GAD = len(GAD_IDS)

CHANNEL_NAMES_LIST = list(df_stats.index)  # 23 channels in canonical order
CH_TO_IDX = {ch: i for i, ch in enumerate(CHANNEL_NAMES_LIST)}

def load_subject_activation(subject_id, task, group):
    """Per-channel STD of concatenated HbO trials. Returns (23,) array."""
    path = DATA_DIR / task / group / subject_id / 'hbo'
    files = sorted(os.listdir(path))
    concat = np.concatenate([np.load(path / f) for f in files], axis=1)
    return concat.std(axis=1)

print("Loading per-subject HbO STD activations...")
hc_activ  = np.array([load_subject_activation(s, TASK, 'healthy') for s in HC_IDS])
gad_activ = np.array([load_subject_activation(s, TASK, 'anxiety') for s in GAD_IDS])
print(f"  HC : {hc_activ.shape}")
print(f"  GAD: {gad_activ.shape}")

# ── Plotting helpers ──
HC_COLOR  = '#5577CC'
GAD_COLOR = '#E74C3C'
DOT_HC    = '#3A5599'   # darker than the bar for visibility
DOT_GAD   = '#A53026'

def plot_one_channel(ax, ch, show_ylabel=False, show_xlabel=True, title_pad=8):
    row = df_stats.loc[ch]
    ch_idx = CH_TO_IDX[ch]
    hc_vals  = hc_activ[:, ch_idx]
    gad_vals = gad_activ[:, ch_idx]

    means = [hc_vals.mean(),  gad_vals.mean()]
    sems  = [hc_vals.std(ddof=1) / np.sqrt(N_HC),
             gad_vals.std(ddof=1) / np.sqrt(N_GAD)]

    bar_w = 0.55
    ax.bar([0, 1], means, yerr=sems,
            width=bar_w, color=[HC_COLOR, GAD_COLOR],
            edgecolor='black', linewidth=0.7, capsize=5,
            error_kw=dict(elinewidth=1.0, ecolor='black'), zorder=2)

    rng = np.random.default_rng(seed=ch_idx)
    jitter_hc  = rng.uniform(-0.18, 0.18, size=N_HC)
    jitter_gad = rng.uniform(-0.18, 0.18, size=N_GAD)
    ax.scatter(jitter_hc, hc_vals, s=14, color=DOT_HC,
                edgecolor='white', linewidth=0.6, alpha=0.85, zorder=3)
    ax.scatter(1 + jitter_gad, gad_vals, s=14, color=DOT_GAD,
                edgecolor='white', linewidth=0.6, alpha=0.85, zorder=3)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['HC', 'GAD'] if show_xlabel else ['', ''], fontsize=10)
    if show_ylabel:
        ax.set_ylabel("HbO concentration (a.u.)", fontsize=10, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='y', labelsize=9)

    # Zoom y-axis: use data range with small padding (no wasted bottom space).
    all_vals = np.concatenate([hc_vals, gad_vals])
    y_lo = max(0.0, all_vals.min() * 0.98)
    y_hi = all_vals.max() * 1.10
    ax.set_ylim(y_lo, y_hi)

    # Significance bracket + marker
    p_fdr = float(row['p_fdr']); sig_fdr = bool(row['sig_fdr'])
    d = float(row['d'])
    y_bar = all_vals.max()
    y_top = y_bar + 0.04 * (y_hi - y_lo)
    y_mark = y_bar + 0.06 * (y_hi - y_lo)
    ax.plot([0, 0, 1, 1], [y_top, y_top + 0.01*(y_hi-y_lo),
                            y_top + 0.01*(y_hi-y_lo), y_top],
             color='black', linewidth=0.9, zorder=4)
    mark = "*" if sig_fdr else "ns"
    ax.text(0.5, y_mark, mark,
             ha='center', va='bottom', fontsize=14 if sig_fdr else 10,
             fontweight='bold' if sig_fdr else 'normal',
             color='black' if sig_fdr else '#888888', zorder=4)

    # Title with stats in subtitle (compact, no internal annotation)
    title = (f"{ch}  ({primary_ba(ch)})\n"
             f"p$_\\mathrm{{FDR}}$={p_fdr:.3f}, $d$={d:+.2f}")
    ax.set_title(title, fontsize=10, fontweight='bold', pad=title_pad)

# ── Save individual panels ──
for ch in top5_channels:
    fig_s, ax_s = plt.subplots(figsize=(3.4, 4.2))
    plot_one_channel(ax_s, ch, show_ylabel=True, show_xlabel=True, title_pad=10)
    out_s = FIG_DIR / f"fig_top5_bar_{ch}.png"
    fig_s.savefig(str(out_s), dpi=300, bbox_inches='tight', facecolor='white')
    fig_s.savefig(str(out_s.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
    plt.close(fig_s)
    print("Saved:", out_s.name)

# ── Composite 1×5 ──
fig, axes = plt.subplots(1, 5, figsize=(14, 4.2))
for i, (ax, ch) in enumerate(zip(axes, top5_channels)):
    plot_one_channel(ax, ch, show_ylabel=(i == 0), show_xlabel=True, title_pad=10)
fig.suptitle("Top-5 XAI channels — HC vs GAD HbO concentration (* = FDR-significant)",
              fontsize=12, fontweight='bold', y=1.01)
plt.subplots_adjust(wspace=0.35, top=0.85)

out = FIG_DIR / 'fig_top5_hbo_bars.png'
fig.savefig(str(out), dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(str(out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
plt.close(fig)
print("Saved:", out.name)

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


Saved: fig_top5_hbo_bars.png


In [9]:
# ─── Section 4: Locked-in colormap (YlOrRd) — re-render all-23 + top-5 overlays ──
# Output: fig_xai_brain_overlay_all23.{png,svg} + fig_xai_brain_overlay_top5.{png,svg}
# Layout: 2x2 grid — A Frontal / B Dorsal / C Left lateral / D Right lateral.

LOCKED_CMAP = 'YlOrRd'

def compose_4view_grid(stem, title, out_name):
    """Read 4 INTERM PNGs ({stem}_{view}.png) → 2x2 grid with A/B/C/D labels."""
    view_titles = {
        'rostral': 'A — Frontal',
        'dorsal':  'B — Dorsal',
        'lat_lh':  'C — Left lateral',
        'lat_rh':  'D — Right lateral',
    }
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, (view, label) in zip(axes.flat, view_titles.items()):
        img = mpimg.imread(str(INTERM / f"{stem}_{view}.png"))
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_title(label, fontsize=13, fontweight='bold', loc='left', pad=4)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.995)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.01,
                        wspace=0.04, hspace=0.06)
    out = FIG_DIR / out_name
    fig.savefig(str(out), dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(str(out.with_suffix('.svg')), bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print("Saved:", out.name)

# Render all-23 (4 views) with YlOrRd
print(f"Rendering all-23 overlay (cmap={LOCKED_CMAP})...")
render_overlay_4views(values_lh_all, values_rh_all,
                       stem='all23', cmap=LOCKED_CMAP, threshold_pct=85)
compose_4view_grid(stem='all23',
                    title='XAI channel-importance overlay — all 23 channels (HbO LOSO mt2)',
                    out_name='fig_xai_brain_overlay_all23.png')

# Render top-5 (4 views) with YlOrRd
print(f"Rendering top-5 overlay (cmap={LOCKED_CMAP})...")
render_overlay_4views(values_lh_top5, values_rh_top5,
                       stem='top5', cmap=LOCKED_CMAP, threshold_pct=85)
compose_4view_grid(stem='top5',
                    title='XAI channel-importance overlay — top-5 channels (HbO LOSO mt2)',
                    out_name='fig_xai_brain_overlay_top5.png')

print("Done — locked colormap:", LOCKED_CMAP)

Rendering all-23 overlay (cmap=YlOrRd)...


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


...

findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


findfont: Font family 'Helvetica' not found.


Saved: fig_xai_brain_overlay_top5.png
Done — locked colormap: YlOrRd
